# 03. Treinamento do LSTM Autoencoder

Este notebook demonstra o treinamento do modelo LSTM Autoencoder no conjunto de treino normal, usando Keras callbacks.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

## 1. Arquitetura do Modelo

In [2]:
from src.model.architecture import build_lstm_autoencoder

model = build_lstm_autoencoder(timesteps=72, n_features=7, variant='classic')
model.summary()

Model: "hexclima_classic"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 72, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_lstm_1 (LSTM)               │ (None, 72, 128)        │        69,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_lstm_2 (LSTM)               │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat (RepeatVector)           │ (None, 72, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_lstm_1 (LSTM)               │ (None, 72, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_lstm_2 (LSTM)               │ (None, 72, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (TimeDistributed)        │ (None, 72, 7)          │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 245,671 (959.65 KB)

 Trainable params: 245,671 (959.65 KB)

 Non-trainable params: 0 (0.00 B)

## 2. Executando o Loop de Treinamento e Calibração dos Thresholds

In [3]:
from src.model.train import run_training

# Executa treinamento completo e calibração dos thresholds sazonais
run_training()

Carregando bases processadas...
Gerando sequencias temporais (future_steps=12)...
X_train shape: (14509, 72, 9)
Shapes das sequencias - X_train: (14509, 72, 9), Y_train_fore: (14509, 12, 9), X_val_normal: (2676, 72, 9)
Construindo o LSTM Autoencoder...


Model: "hexclima_joint_classic"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 72, 9)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm_1 (LSTM)   │ (None, 72, 128)   │     70,656 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm_2 (LSTM)   │ (None, 64)        │     49,408 │ enc_lstm_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bottleneck (Dense)  │ (None, 32)        │      2,080 │ enc_lstm_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat              │ (None, 72, 32)    │          0 │ bottleneck[0][0]  │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_fore         │ (None, 12, 32)    │          0 │ bottleneck[0][0]  │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_1 (LSTM)   │ (None, 72, 64)    │     24,832 │ repeat[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_fore_lstm_1     │ (None, 12, 64)    │     24,832 │ repeat_fore[0][0] │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_2 (LSTM)   │ (None, 72, 128)   │     98,816 │ dec_lstm_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_fore_lstm_2     │ (None, 12, 128)   │     98,816 │ dec_fore_lstm_1[… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_recon        │ (None, 72, 9)     │      1,161 │ dec_lstm_2[0][0]  │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_fore         │ (None, 12, 9)     │      1,161 │ dec_fore_lstm_2[… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 371,762 (1.42 MB)

 Trainable params: 371,762 (1.42 MB)

...

227/227 - 57s - 253ms/step - loss: 0.8357 - output_fore_loss: 0.4057 - output_recon_loss: 0.4300 - val_loss: 1.2074 - val_output_fore_loss: 0.6041 - val_output_recon_loss: 0.6029 - learning_rate: 0.0010
Epoch 8/50

Epoch 8: val_loss improved from 1.20742 to 1.19861, saving model to models/lstm_ae_rs_v1.h5


227/227 - 59s - 260ms/step - loss: 0.8178 - output_fore_loss: 0.3962 - output_recon_loss: 0.4218 - val_loss: 1.1986 - val_output_fore_loss: 0.5852 - val_output_recon_loss: 0.6132 - learning_rate: 0.0010
Epoch 9/50

Epoch 9: val_loss did not improve from 1.19861
227/227 - 79s - 349ms/step - loss: 0.8108 - output_fore_loss: 0.3923 - output_recon_loss: 0.4187 - val_loss: 1.2124 - val_output_fore_loss: 0.5999 - val_output_recon_loss: 0.6121 - learning_rate: 0.0010
Epoch 10/50

Epoch 10: val_loss did not improve from 1.19861
227/227 - 55s - 244ms/step - loss: 0.7977 - output_fore_loss: 0.3853 - output_recon_loss: 0.4123 - val_loss: 1.2046 - val_output_fore_loss: 0.5599 - val_output_recon_loss: 0.6447 - learning_rate: 0.0010
Epoch 11/50

Epoch 11: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 11: val_loss did not improve from 1.19861
227/227 - 44s - 194ms/step - loss: 0.7898 - output_fore_loss: 0.3802 - output_recon_loss: 0.4095 - val_loss: 1.2617 - val_output_for

227/227 - 29s - 129ms/step - loss: 0.7601 - output_fore_loss: 0.3686 - output_recon_loss: 0.3915 - val_loss: 1.1730 - val_output_fore_loss: 0.5753 - val_output_recon_loss: 0.5975 - learning_rate: 5.0000e-04
Epoch 13/50

Epoch 13: val_loss did not improve from 1.17297
227/227 - 31s - 135ms/step - loss: 0.7534 - output_fore_loss: 0.3640 - output_recon_loss: 0.3896 - val_loss: 1.1832 - val_output_fore_loss: 0.5614 - val_output_recon_loss: 0.6219 - learning_rate: 5.0000e-04
Epoch 14/50

Epoch 14: val_loss did not improve from 1.17297
227/227 - 28s - 123ms/step - loss: 0.7468 - output_fore_loss: 0.3615 - output_recon_loss: 0.3853 - val_loss: 1.1845 - val_output_fore_loss: 0.5871 - val_output_recon_loss: 0.5971 - learning_rate: 5.0000e-04
Epoch 15/50

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 15: val_loss did not improve from 1.17297
227/227 - 28s - 124ms/step - loss: 0.7429 - output_fore_loss: 0.3596 - output_recon_loss: 0.3834 - val_loss: 1.1778 - 

227/227 - 41s - 181ms/step - loss: 0.7319 - output_fore_loss: 0.3551 - output_recon_loss: 0.3769 - val_loss: 1.1639 - val_output_fore_loss: 0.5709 - val_output_recon_loss: 0.5928 - learning_rate: 2.5000e-04
Epoch 17/50

Epoch 17: val_loss did not improve from 1.16387
227/227 - 31s - 136ms/step - loss: 0.7288 - output_fore_loss: 0.3539 - output_recon_loss: 0.3750 - val_loss: 1.1941 - val_output_fore_loss: 0.5889 - val_output_recon_loss: 0.6049 - learning_rate: 2.5000e-04
Epoch 18/50

Epoch 18: val_loss improved from 1.16387 to 1.15906, saving model to models/lstm_ae_rs_v1.h5


227/227 - 30s - 134ms/step - loss: 0.7254 - output_fore_loss: 0.3517 - output_recon_loss: 0.3737 - val_loss: 1.1591 - val_output_fore_loss: 0.5689 - val_output_recon_loss: 0.5899 - learning_rate: 2.5000e-04
Epoch 19/50

Epoch 19: val_loss did not improve from 1.15906
227/227 - 28s - 123ms/step - loss: 0.7220 - output_fore_loss: 0.3501 - output_recon_loss: 0.3718 - val_loss: 1.1768 - val_output_fore_loss: 0.5873 - val_output_recon_loss: 0.5891 - learning_rate: 2.5000e-04
Epoch 20/50

Epoch 20: val_loss did not improve from 1.15906
227/227 - 27s - 119ms/step - loss: 0.7197 - output_fore_loss: 0.3494 - output_recon_loss: 0.3702 - val_loss: 1.1919 - val_output_fore_loss: 0.5954 - val_output_recon_loss: 0.5961 - learning_rate: 2.5000e-04
Epoch 21/50

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 21: val_loss did not improve from 1.15906
227/227 - 28s - 123ms/step - loss: 0.7175 - output_fore_loss: 0.3482 - output_recon_loss: 0.3692 - val_loss: 1.1672 - 

## 3. Visualizando os Thresholds Sazonais Salvos

In [4]:
import json

with open('../models/thresholds_rs_v1.json', 'r') as f:
    thresholds = json.load(f)
    
print(json.dumps(thresholds, indent=4))

{
    "global_p95": 1.6715719470685717,
    "global_p97": 1.8010525571891254,
    "global_p99": 2.547498174044984,
    "seasonal": {
        "verao": 0.6316015965956875,
        "outono": 1.8010525571891254,
        "inverno": 1.8010525571891254,
        "primavera": 2.068593526984606
    }
}
